# Experiment 1: fixed Sundays

**Question:** if every Sunday in the history has the same constant temperature, does TiRex-2 notice, and
continue that pattern when the forecast day is a Sunday?

The model has no calendar, it only sees hourly values. So "notice" means: does it pick up a pattern that
repeats every 7 days?

**Setup**
- History: 4 weeks (672 h), so it contains exactly 4 Sundays. Forecast: 24 h, starting at 00:00 UTC.
- **Forecast dates: 40 Sundays spread over the whole year**, each with a control forecast for the **Wednesday**
  of the same week (same manipulation, but the forecast day is not a Sunday).
- **20 random points per date** (different points for each date) = 800 cases per setup.
- Sunday value: **10 °C**, **20 °C** (flat: no day-night cycle), **50 °C** (impossible).
- Input, two versions:
  - **without time feature**: temperature history only
  - **with Sunday flag**: plus a 0/1 feature that marks Sunday hours, for the history *and* the forecast day
    (a future covariate: known in advance, like a calendar)
- Every forecast is also run with the **original** history (same input version), as the reference.

**Measure: score (0-100 %)**, see below. In short: how far the forecast moved towards the fixed Sunday value.


In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp1")
FIG.mkdir(parents=True, exist_ok=True)
FIG.mkdir(exist_ok=True)
C, H = 672, 24  # history hours, forecast hours
N_DATES, N_POINTS = 40, 20
QUICK = bool(os.environ.get("QUICK"))
if QUICK:  # quick code test
    N_DATES, N_POINTS, FIG = 2, 2, Path("/tmp")
VALUES = [10, 20, 50]  # fixed Sunday temperature (°C)
INPUTS = {"without time feature": False, "with Sunday flag": True}
DAYS = ["Sunday", "Wednesday"]
BLUE, ORANGE, GREY = "#1f77b4", "#ff7f0e", "#9e9e9e"
RESULTS = Path(f"/workspace/data/notebooks/results/exp1_forecasts_{N_DATES}x{N_POINTS}.npz")
if QUICK:
    RESULTS = Path("/tmp/exp1_test.npz")
RERUN = False

In [2]:
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
T = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")
Y = T.to_numpy(np.float32)
times = T.index
col = {p: i for i, p in enumerate(T.columns)}

# all Sundays 00 UTC with 4 weeks of history before and the Wednesday forecast inside the data; pick 40 evenly
sundays = [i for i in np.where((times.day_of_week == 6) & (times.hour == 0))[0] if i - C >= 0 and i + 72 + H <= len(times)]
sundays = [sundays[k] for k in np.linspace(0, len(sundays) - 1, N_DATES).round().astype(int)]
rng = np.random.default_rng(0)
cases = pd.DataFrame([(d, p, o) for d, o in enumerate(sundays) for p in rng.choice(T.columns, N_POINTS, replace=False)],
                     columns=["date", "point_id", "o_sun"])
cases["o_wed"] = cases.o_sun + 72
cases["sunday"] = times[cases.o_sun].strftime("%Y-%m-%d")
print(f"{N_DATES} Sundays from {cases.sunday.iloc[0]} to {cases.sunday.iloc[-1]}, {N_POINTS} points each = {len(cases)} cases")

40 Sundays from 2025-11-02 to 2026-09-27, 20 points each = 800 cases


In [3]:
def sunday(t):
    return np.asarray(t.day_of_week == 6)


def history(o, p, value=None, weeks_ago=None):
    """(C,) temperature history before index o at point p. value: all Sundays (or only the Sunday `weeks_ago`) fixed."""
    h = Y[o - C : o, col[p]].copy()
    if value is not None:
        ctx = times[o - C : o]
        if weeks_ago is None:
            h[sunday(ctx)] = value
        else:
            day = times[o].normalize() - pd.Timedelta(weeks=weeks_ago)
            h[np.asarray((ctx >= day) & (ctx < day + pd.Timedelta("1D")))] = value
    return h


model = load_model("NX-AI/TiRex-2", device="cpu")


def forecast(starts, value=None, flag=False, weeks_ago=None):
    """(cases, 9 quantiles, H) forecasts, one per case."""
    ts = [TimeseriesType(torch.from_numpy(history(o, p, value, weeks_ago)[None]), None,
                         torch.from_numpy(sunday(times[o - C : o + H]).astype(np.float32)[None]) if flag else None)
          for o, p in zip(starts, cases.point_id)]
    out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
    return np.stack([q[0] for q in out])


WEEKS_AGO = [1, 2, 3]  # part 2: only one fixed Sunday
res, one = {}, {}
if RESULTS.exists() and not RERUN:
    z = np.load(RESULTS)
    assert list(z["point_id"]) == list(cases.point_id)
    for k in z.files:
        if k.startswith("main|"):
            _, d, i, v = k.split("|")
            res[d, i, None if v == "None" else int(v)] = z[k]
        elif k.startswith("one|"):
            _, w, v = k.split("|")
            one[int(w), int(v)] = z[k]
    print(f"loaded saved forecasts from {RESULTS.name}")
else:
    t0 = time.time()
    for day in DAYS:
        starts = cases.o_sun if day == "Sunday" else cases.o_wed
        for inp, flag in INPUTS.items():
            for value in [None, *VALUES]:
                res[day, inp, value] = forecast(starts, value, flag)
            print(f"{day:9s} {inp:21s} done after {time.time() - t0:.0f} s")
    for w in WEEKS_AGO:
        for v in VALUES:
            one[w, v] = forecast(cases.o_sun, v, False, w)
        print(f"one Sunday {w} week(s) ago done after {time.time() - t0:.0f} s")
    np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), sunday=cases.sunday.to_numpy().astype("U"),
                        truth_sun=np.stack([Y[o : o + H, col[p]] for o, p in zip(cases.o_sun, cases.point_id)]),
                        **{f"main|{d}|{i}|{v}": q for (d, i, v), q in res.items()},
                        **{f"one|{w}|{v}": q for (w, v), q in one.items()})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp1_forecasts_40x20.npz


## Score for all cases

**Score:** for each forecast hour, how far the forecast moved from its reference (same input, original
history) towards the fixed Sunday value, capped between 0 % and 100 %. Hours where the reference is already
within 1 °C of the Sunday value are skipped (nothing to move). Score of a case = average over its hours.
0 % = the model ignores the fixed Sundays, 100 % = it follows them every hour. One dot = one case (one point on
one date).

In [4]:
def score_from(ref, new, value):
    """Per case: mean over hours of the share of the way moved towards the Sunday value (0-100 %)."""
    ref, new = ref[:, 4], new[:, 4]
    gap = value - ref
    return 100 * np.nanmean(np.clip((new - ref) / np.where(np.abs(gap) >= 1, gap, np.nan), 0, 1), axis=1)


def score(day, inp, value):
    return score_from(res[day, inp, None], res[day, inp, value], value)


scores = pd.DataFrame([
    {"case": k, "point_id": cases.point_id[k], "sunday": cases.sunday[k], "forecast day": day, "input": inp,
     "Sunday value": f"{v} °C", "score": s}
    for day in DAYS for inp in INPUTS for v in VALUES for k, s in enumerate(score(day, inp, v))
])
scores.groupby(["forecast day", "input", "Sunday value"]).score.mean().unstack().round(0)

Sunday value                       10 °C  20 °C  50 °C
forecast day input                                    
Sunday       with Sunday flag       72.0   77.0   98.0
             without time feature   52.0   66.0   94.0
Wednesday    with Sunday flag        6.0    6.0    1.0
             without time feature    6.0    6.0    1.0

In [5]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["Forecast for a Sunday", "Control: forecast for a Wednesday"])
for c, day in enumerate(DAYS, start=1):
    for inp, color in [("without time feature", GREY), ("with Sunday flag", ORANGE)]:
        s = scores[(scores["forecast day"] == day) & (scores.input == inp)]
        fig.add_trace(go.Box(
            x=s["Sunday value"], y=s.score, name=inp, legendgroup=inp, showlegend=c == 1,
            marker={"color": color, "size": 2, "opacity": 0.5}, line={"color": color}, boxpoints="all", jitter=0.5, pointpos=0,
            customdata=np.c_[s.point_id, s.sunday],
            hovertemplate="point %{customdata[0]}, %{customdata[1]}: %{y:.0f} %<extra>" + inp + "</extra>"), row=1, col=c)
fig.update_layout(
    boxmode="group", height=500, template="plotly_dark", legend={"orientation": "h", "y": -0.15},
    title=f"Does the forecast follow the fixed Sundays?<br><sup>{len(cases)} cases ({N_DATES} dates over the year × {N_POINTS} points), "
          "one dot each; 4 fixed Sundays in the 4-week history. 0 % = ignores them, 100 % = follows them</sup>")
fig.update_yaxes(title_text="score (%)", range=[-3, 103], row=1, col=1)
fig.update_xaxes(title_text="fixed Sunday value")
fig.write_html(FIG / "exp1_scores_all_points.html", include_plotlyjs="cdn")
fig.show()

## Does it hold across the year?

Mean score of the 20 points of each date, for the Sunday forecasts. If the effect were specific to one week or
season, the lines would jump; flat lines = systematic over the year.

In [6]:
fig = go.Figure()
for v, dash in zip(VALUES, ["solid", "dash", "dot"]):
    for inp, color in [("without time feature", GREY), ("with Sunday flag", ORANGE)]:
        s = scores[(scores["forecast day"] == "Sunday") & (scores.input == inp) & (scores["Sunday value"] == f"{v} °C")]
        m = s.groupby("sunday").score.mean()
        fig.add_trace(go.Scatter(x=pd.to_datetime(m.index), y=m.values, mode="lines+markers", name=f"{v} °C, {inp}",
                                 line={"color": color, "dash": dash, "width": 2}, marker={"size": 5, "color": color},
                                 hovertemplate="%{x|%d %b %Y}: %{y:.0f} %<extra>" + f"{v} °C, {inp}" + "</extra>"))
fig.update_layout(height=460, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"Score per forecast date (mean of {N_POINTS} points), Sunday forecasts",
                  xaxis={"title": "forecast Sunday"}, yaxis={"title": "score (%)", "range": [-3, 103]})
fig.write_html(FIG / "exp1_score_by_date.html", include_plotlyjs="cdn")
fig.show()

## One case, all setups

The case with the median score (10 °C, without time feature), so the example is typical, not the most striking.
Each panel: last 8 days of history (the previous fixed Sunday is shaded) and the Sunday forecast.
Blue = original history and its forecast, orange = fixed Sundays and its forecast, white = what happened,
dashed = the fixed Sunday value. Zoom with the mouse; double-click to reset.

In [7]:
per_case = score("Sunday", "without time feature", 10)
j = int(np.argsort(per_case)[len(per_case) // 2])
pid, o = cases.point_id[j], cases.o_sun[j]
win = times[o - 8 * 24 : o + H]
fut_t = times[o : o + H]
print(f"point {pid}, Sunday {cases.sunday[j]}, score at 10 °C without time feature: {per_case[j]:.0f}")

fig = make_subplots(rows=len(VALUES), cols=len(INPUTS), shared_xaxes=True, vertical_spacing=0.07, horizontal_spacing=0.06,
                    subplot_titles=[f"Sunday = {v} °C, {inp}" for v in VALUES for inp in INPUTS])
for r, value in enumerate(VALUES, start=1):
    for c, inp in enumerate(INPUTS, start=1):
        first = r == 1 and c == 1
        hist_o, hist_f = history(o, pid)[-8 * 24 :], history(o, pid, value)[-8 * 24 :]
        hist_t = win[: 8 * 24]
        for d in pd.date_range(win[0], win[-1], freq="D"):
            if d.day_of_week == 6:
                fig.add_vrect(x0=d, x1=d + pd.Timedelta("1D"), fillcolor=ORANGE, opacity=0.08, line_width=0, row=r, col=c)
        traces = [
            go.Scatter(x=hist_t, y=hist_o, name="history, original", line={"color": BLUE, "width": 1}),
            go.Scatter(x=hist_t, y=hist_f, name="history, fixed Sundays", line={"color": ORANGE, "width": 1.2}),
            go.Scatter(x=fut_t, y=Y[o : o + H, col[pid]], name="what happened", line={"color": "white", "width": 1.5}),
            go.Scatter(x=[fut_t[0], fut_t[-1]], y=[value, value], name="fixed Sunday value", line={"color": ORANGE, "dash": "dash", "width": 1}),
        ]
        for key, color, label in [(None, BLUE, "original"), (value, ORANGE, "fixed Sundays")]:
            q = res["Sunday", inp, key][j]
            traces += [
                go.Scatter(x=np.r_[fut_t, fut_t[::-1]], y=np.r_[q[8], q[0][::-1]], mode="lines", fill="toself", fillcolor=color, opacity=0.18,
                           line={"width": 0}, name=f"10-90 % band, {label}", hoverinfo="skip"),
                go.Scatter(x=fut_t, y=q[4], name=f"forecast, {label}", line={"color": color, "width": 2.2}),
            ]
        for tr in traces:
            tr.update(legendgroup=tr.name, showlegend=first)
            fig.add_trace(tr, row=r, col=c)
        fig.add_vline(x=fut_t[0], line={"color": "grey", "dash": "dot", "width": 1}, row=r, col=c)
    fig.update_yaxes(title_text="T2M [°C]", row=r, col=1)
fig.update_xaxes(tickformat="%a %d %b\n%H:%M")
fig.update_layout(height=900, template="plotly_dark", legend={"orientation": "h", "y": -0.08},
                  title=f"Point {pid}, Sunday {cases.sunday[j]}: Sunday forecast with fixed Sundays in the history")
fig.write_html(FIG / "exp1_example_point.html", include_plotlyjs="cdn")
fig.show()

point 1750, Sunday 2026-09-13, score at 10 °C without time feature: 55


## What decides whether a case follows the fixed Sundays?

Idea: the more the fixed Sunday stands out from the real Sundays, the more the model follows it.

**Size of the change** (x-axis): how much the manipulation changed the history, i.e. the mean absolute
difference between the fixed value and the real temperature over all Sunday hours of the 4-week history (°C).
It covers both the level (e.g. 10 °C vs 25 °C) and the shape (a flat line vs a normal day-night cycle).

One panel per Sunday value, one dot per case (grey = without, orange = with Sunday flag). The line is the median
score of the cases in each 2 °C bin, the band covers the middle half of them.

In [8]:
change = {v: np.array([np.abs(v - history(o, p)[sunday(times[o - C : o])]).mean() for o, p in zip(cases.o_sun, cases.point_id)])
          for v in VALUES}
dots = scores[scores["forecast day"] == "Sunday"].copy()
dots["change"] = [change[int(v.split()[0])][k] for v, k in zip(dots["Sunday value"], dots.case)]

LIGHT = {GREY: "rgba(158,158,158,0.18)", ORANGE: "rgba(255,127,14,0.18)"}
vals = [f"{v} °C" for v in VALUES]
fig = make_subplots(rows=1, cols=len(vals), shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=[f"Sunday = {v}" for v in vals])
for c, val in enumerate(vals, start=1):
    for inp, color in [("without time feature", GREY), ("with Sunday flag", ORANGE)]:
        s = dots[(dots.input == inp) & (dots["Sunday value"] == val)]
        first = c == 1
        fig.add_trace(go.Scatter(
            x=s.change, y=s.score, mode="markers", name=f"one case, {inp}", legendgroup=f"dots {inp}", showlegend=first,
            marker={"color": color, "size": 3, "opacity": 0.2}, customdata=np.c_[s.point_id, s.sunday],
            hovertemplate="point %{customdata[0]}, %{customdata[1]}<br>change %{x:.1f} °C, score %{y:.0f} %<extra>" + inp + "</extra>"),
            row=1, col=c)
        b = s.groupby((s.change // 2) * 2 + 1).score
        st = pd.DataFrame({"q25": b.quantile(0.25), "med": b.median(), "q75": b.quantile(0.75), "n": b.size()})
        st = st[st.n >= 10]  # skip bins with fewer than 10 cases
        fig.add_trace(go.Scatter(x=np.r_[st.index, st.index[::-1]], y=np.r_[st.q75, st.q25[::-1]], mode="lines", fill="toself",
                                 fillcolor=LIGHT[color], line={"width": 0}, hoverinfo="skip", showlegend=False,
                                 legendgroup=f"median {inp}"), row=1, col=c)
        fig.add_trace(go.Scatter(
            x=st.index, y=st.med, mode="lines+markers", line={"color": color, "width": 3}, marker={"size": 7, "color": color},
            name=f"median of cases (2 °C bins), {inp}", legendgroup=f"median {inp}", showlegend=first,
            customdata=np.c_[st.n, st.q25, st.q75],
            hovertemplate="change %{x:.0f} ± 1 °C: median %{y:.0f} % (middle half %{customdata[1]:.0f}-%{customdata[2]:.0f} %), "
                          "%{customdata[0]} cases<extra>" + inp + "</extra>"), row=1, col=c)
    fig.update_xaxes(title_text="change to the Sundays (°C)", row=1, col=c)
fig.update_layout(
    height=480, template="plotly_dark", legend={"orientation": "h", "y": -0.25},
    title="Does the model follow the fixed Sundays more when they differ more from the real Sundays?<br>"
          f"<sup>Forecast for a Sunday, {len(cases)} cases per panel. x: mean absolute difference between the fixed value and the "
          "real Sunday temperatures in the history. Line: median score per 2 °C bin, band: middle half of the cases</sup>")
fig.update_yaxes(title_text="score (%)", range=[-3, 103], row=1, col=1)
fig.write_html(FIG / "exp1_score_vs_change.html", include_plotlyjs="cdn")
fig.show()

# Part 2: only one fixed Sunday

**Question:** if only **one** Sunday in the history is fixed (the other three stay real), does that one-off
anomaly leak into the Sunday forecast? With a single occurrence there is no repeating pattern to continue.

- Same Sunday forecasts and cases as above, **without time feature**.
- One Sunday fixed to 10 / 20 / 50 °C: **1, 2 or 3 weeks** before the forecast.
- Same score (reference = same forecast with the original history), compared with **all 4 Sundays fixed**.

In [9]:
ref = res["Sunday", "without time feature", None]
rows = [{"setup": f"1 Sunday, {w} week{'s' if w > 1 else ''} ago", "Sunday value": f"{v} °C", "point_id": cases.point_id[k],
         "sunday": cases.sunday[k], "score": s}
        for w in WEEKS_AGO for v in VALUES for k, s in enumerate(score_from(ref, one[w, v], v))]
rows += [{"setup": "all 4 Sundays", "Sunday value": r["Sunday value"], "point_id": r.point_id, "sunday": r.sunday, "score": r.score}
         for _, r in scores[(scores["forecast day"] == "Sunday") & (scores.input == "without time feature")].iterrows()]
one_scores = pd.DataFrame(rows)
SETUPS = [f"1 Sunday, {w} week{'s' if w > 1 else ''} ago" for w in WEEKS_AGO] + ["all 4 Sundays"]
one_scores.groupby(["setup", "Sunday value"]).score.mean().unstack().reindex(SETUPS).round(0)

Sunday value,10 °C,20 °C,50 °C
setup,,,
"1 Sunday, 1 week ago",2.0,2.0,1.0
"1 Sunday, 2 weeks ago",2.0,2.0,1.0
"1 Sunday, 3 weeks ago",2.0,2.0,1.0
all 4 Sundays,52.0,66.0,94.0


In [10]:
SHADES = ["#fdd0a2", "#fdae6b", "#f16913", "#d94801"]  # light = one old Sunday ... strong = all 4 Sundays
fig = go.Figure()
for setup, color in zip(SETUPS, SHADES):
    s = one_scores[one_scores.setup == setup]
    fig.add_trace(go.Box(
        x=s["Sunday value"], y=s.score, name=setup, marker={"color": color, "size": 2, "opacity": 0.5}, line={"color": color},
        boxpoints="all", jitter=0.5, pointpos=0, customdata=np.c_[s.point_id, s.sunday],
        hovertemplate="point %{customdata[0]}, %{customdata[1]}: %{y:.0f} %<extra>" + setup + "</extra>"))
fig.update_layout(
    boxmode="group", height=480, template="plotly_dark", legend={"orientation": "h", "y": -0.15},
    title="One fixed Sunday vs all four: does a one-off anomaly leak into the Sunday forecast?<br>"
          f"<sup>{len(cases)} cases ({N_DATES} dates × {N_POINTS} points), one dot each, without time feature. "
          "0 % = ignores it, 100 % = follows it</sup>",
    xaxis={"title": "fixed Sunday value"}, yaxis={"title": "score (%)", "range": [-3, 103]})
fig.write_html(FIG / "exp1_one_vs_all_sundays.html", include_plotlyjs="cdn")
fig.show()

## Reusing the results

```python
z = np.load("/workspace/data/notebooks/results/exp1_forecasts_40x20.npz")
z["point_id"], z["sunday"]                         # one entry per case (point, forecast Sunday)
z["truth_sun"]                                     # (cases, 24) what happened on the Sunday
z["main|Sunday|with Sunday flag|10"]               # (cases, 9 quantiles, 24 h); day Sunday/Wednesday, value None/10/20/50
z["one|2|10"]                                      # part 2: only the Sunday 2 weeks ago fixed to 10 °C (Sunday forecast)
```
(the Wednesday forecast of case k starts 3 days after its Sunday)